# Enchilada: minimal interface demo

This notebook walks through the `L1Data` / `Block` / `Wheel` contract using `EchoBlock`, a no-op block that just prints what the Wheel hands it. No real waveform model, no MCMC — just enough to see the plumbing work.

## 1. Build the observed `L1Data`

One frozen object holds both the TDI arrays and the run settings everyone in this run agrees on.

In [ ]:
import numpy as np

from enchilada import L1Data, Wheel
from enchilada.testing import EchoBlock

rng = np.random.default_rng(0)
n_samples = 1024
channels = ("A", "E", "T")

observed = L1Data(
    tdi={ch: rng.standard_normal(n_samples) for ch in channels},
    sample_rate=0.1,
    channels=channels,  # n_samples read off the arrays
    tdi_generation="2.0",
    observable="fractional_frequency",
)  # epoch defaults to 0.0
observed

## 2. Long names and short shadows

Every derived quantity has two spellings. Use whichever reads better.

In [ ]:
print(f"observation_time = {observed.observation_time} s     (Tobs = {observed.Tobs})")
print(f"sample_rate      = {observed.sample_rate} Hz   (fs   = {observed.fs})")
print(f"sample_interval  = {observed.sample_interval} s     (dt   = {observed.dt})")
print(f"n_samples        = {observed.n_samples}        (N    = {observed.N})")
print(f"freq_resolution  = {observed.frequency_resolution} Hz   (df   = {observed.df})")
print(f"nyquist          = {observed.nyquist_frequency} Hz   (fny  = {observed.fny})")
print(f"epoch            = {observed.epoch} s     (t0   = {observed.t0})")

In [ ]:
L1Data.aliases()

## 3. Typo catcher

Common misspellings point at the canonical spelling instead of failing silently.

In [ ]:
try:
    _ = observed.T_obs
except AttributeError as e:
    print(e)

The Wheel keeps the pristine data plus a **ledger** of each block's current model, and hands each block the data minus *every other* block — never its own. The block fits that, subtracts its new model, and returns; the Wheel reads its new ledger entry off the difference. So there is no "add-back" to forget. Blocks own all of their internal state, so hold on to the objects you register if you want to read anything back afterwards.

In [ ]:
ucb = EchoBlock(name="ucb")
mbhb = EchoBlock(name="mbhb")

wheel = Wheel(observed)
wheel.add(ucb)
wheel.add(mbhb)

wheel.run(n_cycles=3)

## 4. Block state stays with the blocks

The Wheel holds the pristine data and the ledger — one model per block — and nothing else about the fit. Anything else — parameters, chains, update counters — lives on the block objects themselves; ask them directly. `wheel.residual()` is the full residual (data minus every model) and `wheel.residual(exclude=name)` is what that block sees; `wheel.contribution(name)` is its ledger entry.

In [ ]:
print(f"ucb updates: {ucb.updates}, mbhb updates: {mbhb.updates}")

full = wheel.residual()  # observed minus every block's model
print(f"full residual RMS on 'A': {np.sqrt(np.mean(full.tdi['A'] ** 2)):.4f}")

## 5. Attaching the constellation ephemeris

Real data comes with the spacecraft positions it was produced with. That ephemeris rides on `L1Data.orbit` so every block builds its response from the *same* constellation — and `L1Data` checks at construction that the ephemeris actually spans the observation, catching epoch mismatches (GPS vs zero-based times) before any sampling starts.

`NumericOrbit` needs the `numeric-orbits` extra (`uv sync --extra numeric-orbits`). Here we tabulate a synthetic circular constellation; for real data use `NumericOrbit.from_hdf5` (LDC/Mojito files) or `NumericOrbit.from_lisaorbits`.

In [ ]:
from dataclasses import replace

from enchilada import NumericOrbit

# a synthetic 40-day ephemeris: three spacecraft on a 1 AU circle
AU = 1.495978707e11
t_grid = np.linspace(0.0, 40 * 86400.0, 200)
pos = np.zeros((3, t_grid.size, 3))  # (spacecraft, time, xyz), ecliptic metres
for sc in range(3):
    # spacecraft spaced ~0.0167 rad apart on the circle -> ~2.5e9 m arms
    ang = 2 * np.pi * t_grid / (365.25 * 86400.0) + sc * 0.0167
    pos[sc, :, 0] = AU * np.cos(ang)
    pos[sc, :, 1] = AU * np.sin(ang)

orbit = NumericOrbit(t_grid, pos)
print(
    f"tabulated span: {orbit.t_range}, L = {orbit.L:.3e} m, fstar = {orbit.fstar:.4f} Hz"
)

observed_with_orbit = replace(observed, orbit=orbit)  # spans the data: fine
print("orbit attached; blocks read residual.orbit instead of building their own")

In [ ]:
# the consistency checks in action: a data span the ephemeris does not
# cover is rejected at construction, and out-of-span position queries refuse
# to extrapolate
try:
    replace(observed, orbit=orbit, epoch=39.9 * 86400.0)
except ValueError as e:
    print(f"span check: {e}\n")

try:
    orbit.positions(np.array([100 * 86400.0]))
except ValueError as e:
    print(f"extrapolation check: {e}")

## 6. Where next

- **A real (toy) fit** — [`examples/toy_fit.py`](toy_fit.py) runs two conjugate-Gibbs source blocks plus a sampled white-noise block to convergence: the noise contract (`residual.noise_variance`), posterior chains kept on the block objects, and progress via `Wheel.run(..., on_cycle=...)`.
- **Your own block** — implement the two-method `start`/`update` protocol (`src/enchilada/block.py` docstrings are the contract), then run `enchilada.testing.check_block(your_block, toy_observed)` before plugging into a shared campaign.